# shannon_model — pipeline de negocio en Google Colab

Corre el flujo real (extract de notas, impacto en vistas, oportunidades editoriales) en Colab, usando el codigo versionado del repo.

**Distinto de** `notebooks/colab_train.ipynb`: ese notebook solo valida GPU/PyTorch con un train sintetico. Este notebook corre el pipeline de negocio de punta a punta.

Detalle de flags, layout de datos y problemas frecuentes: ver `docs/COLAB.md`.

Orden de ejecucion:
1. Clonar repo + instalar dependencias
2. Montar Drive y copiar datos a `data/raw/` y `data/processed/`
3. Extract (manual, opcional si ya tenes el parquet)
4. Pipelines de negocio: impacto en vistas + oportunidades editoriales
5. Poll loop: re-corre vistas + editorial solo cuando detecta push relevante (`src/`, `scripts/`, `configs/`) — el extract NO se dispara solo
6. Copiar resultados a Drive

## 1. Clonar repo + instalar dependencias

In [ ]:
# Cambia estos valores si el remoto o la rama son distintos
REPO_URL = "https://github.com/isaacgrimaldovo/shannon_model.git"
BRANCH = "main"
REPO_DIR = "shannon_model"

# Si el repo es privado, descomenta y usa un secret de Colab:
# from google.colab import userdata
# TOKEN = userdata.get("GITHUB_TOKEN")
# REPO_URL = f"https://{TOKEN}@github.com/isaacgrimaldovo/shannon_model.git"

In [ ]:
import os
from pathlib import Path

# Clonar en /content (evita shannon_model/shannon_model anidado)
%cd /content
if Path(REPO_DIR).exists():
    %cd {REPO_DIR}
    !git fetch origin
    !git checkout {BRANCH}
    !git pull origin {BRANCH}
else:
    !git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}
    %cd {REPO_DIR}

!pip install -q -r requirements.txt

import sys
sys.path.insert(0, str(Path("src").resolve()))
print("cwd:", os.getcwd())

## 2. Montar Drive y copiar datos

Layout esperado en Drive (armado una sola vez desde tu PC) — ver `docs/COLAB.md` seccion 2:

```
MyDrive/shannon_model_data/
  raw/
    csv_urls/
    html/                    # opcional si vas a reprocess o scrape
    scrape_index.csv         # opcional
    ehm_3months_filtered.xlsx
  processed/
    notes_structured.parquet # si ya extrajiste en local, alcanza con esto + csv_urls
```

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

SRC = "/content/drive/MyDrive/shannon_model_data"
!mkdir -p data/raw data/processed
!cp -r {SRC}/raw/csv_urls data/raw/
!cp {SRC}/processed/notes_structured.parquet data/processed/  # si ya existe

# Solo si vas a extract/reprocess:
# !cp -r {SRC}/raw/html data/raw/
# !cp {SRC}/raw/scrape_index.csv data/raw/
# !cp {SRC}/raw/ehm_3months_filtered.xlsx data/raw/

## 3. Extract (manual)

Entrypoint: `scripts/scrape_news.py`. Escribe `data/processed/notes_structured.parquet`. HTML e indice quedan en `data/raw/`.

**Manual e independiente del poll loop** (seccion 5): fetch+extract puede tardar minutos u horas y depende de red — no tiene sentido re-dispararlo automaticamente en cada push. Corre esta celda a mano cuando necesites datos nuevos o re-parsear HTML existente.

In [ ]:
REPROCESS_ONLY = True  # True: re-parsea html/ ya en disco (rapido, no pega al sitio)
                        # False: fetch + extract desde el sitio (lento, usa --limit para probar)
LIMIT = 50              # solo aplica si REPROCESS_ONLY=False; None = todas las URLs pendientes

if REPROCESS_ONLY:
    !python scripts/scrape_news.py --reprocess \
      --html-dir data/raw/html \
      --index-path data/raw/scrape_index.csv \
      --urls-xlsx data/raw/ehm_3months_filtered.xlsx \
      --structured-path data/processed/notes_structured.parquet
else:
    limit_flag = f"--limit {LIMIT}" if LIMIT is not None else ""
    !python scripts/scrape_news.py {limit_flag} --workers 2 \
      --urls-xlsx data/raw/ehm_3months_filtered.xlsx \
      --html-dir data/raw/html \
      --index-path data/raw/scrape_index.csv \
      --structured-path data/processed/notes_structured.parquet

## 4. Pipelines de negocio

Requiere `data/processed/notes_structured.parquet` + `data/raw/csv_urls/`.

In [ ]:
# Modelo A/B de impacto en vistas + SHAP (+ por categoria)
!python scripts/train_views_model.py --config configs/views_impact.yaml

In [ ]:
# Receta por seccion -> cumplimiento -> mayor oportunidad + KPIs
!python scripts/report_editorial_opportunities.py --config configs/editorial_opportunities.yaml

## 5. Poll loop (auto-run vistas + editorial en cambios push)

Flujo: hace `commit` + `push` desde tu entorno local tocando `src/`, `scripts/` o `configs/`.
Con este loop corriendo en Colab, cada `POLL_INTERVAL` segundos se hace `git pull`;
si detecta un commit nuevo que toca esos paths, dispara `train_views_model.py` +
`report_editorial_opportunities.py` automaticamente (mismos entrypoints que la seccion 4).

**El extract (seccion 3) NO se dispara desde este loop** — queda siempre manual.
Cambios que solo tocan docs/README/notebook tampoco disparan run.

Para parar: boton **Interrupt execution** de Colab — el loop corta limpio, sin dejar los scripts colgados.

In [ ]:
import subprocess
import time

POLL_INTERVAL = 20  # segundos entre cada git pull
WATCH_PATHS = ("src/", "scripts/", "configs/")
VIEWS_CONFIG = "configs/views_impact.yaml"
EDITORIAL_CONFIG = "configs/editorial_opportunities.yaml"


def _current_sha():
    return subprocess.run(
        ["git", "rev-parse", "HEAD"], capture_output=True, text=True, check=True
    ).stdout.strip()


def _relevant_change(old_sha, new_sha):
    diff = subprocess.run(
        ["git", "diff", "--name-only", old_sha, new_sha],
        capture_output=True, text=True, check=True,
    ).stdout.splitlines()
    return any(path.startswith(WATCH_PATHS) for path in diff)


last_sha = _current_sha()
print(f"[poll] loop iniciado, SHA inicial: {last_sha[:8]}")

try:
    while True:
        subprocess.run(["git", "pull"], capture_output=True, text=True)
        new_sha = _current_sha()

        if new_sha == last_sha:
            print(f"[poll] {new_sha[:8]} — sin cambios")
        elif _relevant_change(last_sha, new_sha):
            print(f"[poll] {new_sha[:8]} — cambio relevante, corriendo views + editorial")
            subprocess.run(["python", "scripts/train_views_model.py", "--config", VIEWS_CONFIG], check=False)
            subprocess.run(["python", "scripts/report_editorial_opportunities.py", "--config", EDITORIAL_CONFIG], check=False)
            last_sha = new_sha
        else:
            print(f"[poll] {new_sha[:8]} — cambio detectado pero irrelevante (no toca src/scripts/configs)")
            last_sha = new_sha

        time.sleep(POLL_INTERVAL)
except KeyboardInterrupt:
    print("[poll] interrumpido manualmente, loop detenido")

## 6. Copiar resultados a Drive

In [ ]:
!mkdir -p /content/drive/MyDrive/shannon_model/checkpoints
!cp -r checkpoints/views_impact /content/drive/MyDrive/shannon_model/checkpoints/
!cp -r checkpoints/editorial_opportunities /content/drive/MyDrive/shannon_model/checkpoints/
!cp data/processed/notes_structured.parquet /content/drive/MyDrive/shannon_model_data/processed/